# InfluxDB Benchmarks – Big Experiments

## Libraries

In [ ]:
from fabric import ThreadingGroup, Connection
from chi import server, ssh
import os
import warnings
import time
import sys
sys.path.append('../scripts')
from utilities import *
warnings.filterwarnings('ignore')

## Configurations

In [ ]:
sshUser = "ubuntu"
server_ips = ["64.131.114.155"]
server_local_ips = {}
client_ips = ["64.131.114.143", "64.131.114.145", "64.131.114.146", "64.131.114.147",
              "64.131.114.148", "64.131.114.152", "64.131.114.153", "64.131.114.154"]

storageType = {"HDD": "/HDD", "NVME": "/NVME_RAID-0"}
SERVER_WORKING_DIR = storageType["NVME"]

## Initialize environment

In [ ]:
# These groups are for processing in parallel 
serverConnections = [makeConnection(i, user=sshUser) for i in server_ips]
clientConnections = [makeConnection(i, user=sshUser) for i in client_ips]
allConnections = (serverConnections + clientConnections)
serverGroup = ThreadingGroup.from_connections(serverConnections)
clientGroup = ThreadingGroup.from_connections(clientConnections)
allGroup = ThreadingGroup.from_connections(allConnections)

# Get local IP for server
local_ips = serverGroup.run("hostname -I | cut -d ' ' -f 1", hide=True)
for i in local_ips:
    server_local_ips[i.host] = local_ips[i].stdout.strip()

# Update & Upgrade latest packages
allGroup.run("sudo apt update && sudo apt upgrade -y", hide=True)
rebootGroup(allGroup)

Wait for all server to come online after reboot

In [ ]:
# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

## Datasets

Preparing test datasets on client nodes

In [ ]:
clientGroup.put("../scripts/initData.sh")
clientGroup.run("bash initData.sh MYSTIC@IIT", hide=True);

## Benchmark server (InfluxDB)

Prepare benchmark server

In [ ]:
serverGroup.put("../scripts/_common.sh")
serverGroup.put("../scripts/InfluxDB/initInfluxDBServer.sh")
serverGroup.run(f"sudo chmod -R 777 {SERVER_WORKING_DIR}; bash initInfluxDBServer.sh {SERVER_WORKING_DIR} true", hide=True);

## Benchmark client (XBench)
Prepare benchmark client

In [ ]:
clientGroup.put("../scripts/initXBenchClient.sh")
clientGroup.run("bash initXBenchClient.sh", hide=True)
clientGroup.close()

## Experiments - Settings

In [ ]:
targetThreads = [1, 2, 4, 8]
numClients = [i * len(client_ips) for i in targetThreads]

## Experiment – UNARY INSERT (SEQ)
**Perform experiments**

In [ ]:
runWorkload("INSERT_UNARY_SEQ", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("INSERT_UNARY_SEQ", "InfluxDB", targetThreads, client_ips, "Insert", logScale=True)

## Experiment – UNARY INSERT (RAND)
**Perform experiments**

In [ ]:
runWorkload("INSERT_UNARY_RAND", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("INSERT_UNARY_RAND", "InfluxDB", targetThreads, client_ips, "Insert", logScale=True)

## Experiment – BATCH INSERT (SEQ)
**Perform experiments**

In [ ]:
runWorkload("INSERT_BATCH_SEQ", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("INSERT_BATCH_SEQ", "InfluxDB", targetThreads, client_ips)

## Experiment – BATCH INSERT (RAND)
**Perform experiments**

In [ ]:
runWorkload("INSERT_BATCH_RAND", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("INSERT_BATCH_RAND", "InfluxDB", targetThreads, client_ips)

## Bootstrap Database

Reboot server to a fresh state to avoid becoming non-accessible from public network

In [ ]:
rebootGroup(allGroup)

# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

<div class="alert alert-block alert-info">Note: This is required so that there's actual data for clients to query from</div>

In [ ]:
serverGroup.put("../scripts/restoreDB.sh")

# Restore DB
serverGroup.run(f"bash restoreDB.sh MYSTIC@IIT InfluxDB BENCH_DB {SERVER_WORKING_DIR}", hide=True)

# Keep checking if job is done running in tmux window
checkTmuxJobIsDone(serverGroup);

## Experiment – UNARY QUERY (SEQ)

**Perform experiments**

In [ ]:
runWorkload("QUERY_UNARY_SEQ", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("QUERY_UNARY_SEQ", "InfluxDB", targetThreads, client_ips, "Query", logScale=True)

## Experiment – UNARY QUERY (RAND)

**Perform experiments**

In [ ]:
runWorkload("QUERY_UNARY_RAND", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("QUERY_UNARY_RAND", "InfluxDB", targetThreads, client_ips, "Query", logScale=True)

## Experiment – BATCH QUERY (SEQ)

**Perform experiments**

In [ ]:
runWorkload("QUERY_BATCH_SEQ", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 100, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("QUERY_BATCH_SEQ", "InfluxDB", targetThreads, client_ips, 100)

## Refresh Database

Reboot server to a fresh state to avoid becoming non-accessible from public network

In [ ]:
rebootGroup(allGroup)

# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

## Experiment – MIN AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_MIN_QUERY", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_MIN_QUERY", "InfluxDB", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – MAX AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_MAX_QUERY", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_MAX_QUERY", "InfluxDB", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – SUM AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_SUM_QUERY", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_SUM_QUERY", "InfluxDB", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – AVG AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_AVG_QUERY", targetThreads, server_local_ips[server_ips[0]], 9492, "InfluxDB", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_AVG_QUERY", "InfluxDB", targetThreads, client_ips, "Aggregate", logScale=True)